# Análise de Resistência à Compressão do Concreto

Este notebook apresenta um projeto completo de Machine Learning para prever a resistência à compressão do concreto com base em seus componentes e idade da cura.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configurações de visualização
%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Carregamento e Limpeza dos Dados

In [ ]:
# Carregando o dataset diretamente da fonte oficial UCI
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls'
df = pd.read_excel(url)

# Removendo registros duplicados para garantir a integridade da análise
df = df.drop_duplicates().copy()

print(f'Dataset carregado com sucesso. Dimensões após limpeza: {df.shape}')
display(df.head())

## 2. Divisão dos Dados em Treino e Teste

In [ ]:
# Definição do target e features
target_column = 'Concrete compressive strength(MPa, megapascals) '
X = df.drop(columns=[target_column])
y = df[target_column]

# Divisão clássica 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')

## 3. Treinamento e Comparação Inicial de Modelos

Vamos treinar uma Regressão Linear (Baseline) e uma Árvore de Decisão padrão.

In [ ]:
# Inicialização
linear_model = LinearRegression()
decision_tree = DecisionTreeRegressor(random_state=42)

results = {}

for name, model in [('Regressão Linear', linear_model), ('Árvore de Decisão', decision_tree)]:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}

df_results = pd.DataFrame(results).T
display(df_results)

## 4. Otimização e Avanço com Random Forest

Testaremos o modelo Random Forest com ajuste de hiperparâmetros via GridSearchCV.

In [ ]:
# Grid de parâmetros para Random Forest
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 15, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf_grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=rf_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

rf_grid_search.fit(X_train, y_train)
best_rf_model = rf_grid_search.best_estimator_

# Predição e Métricas
y_pred_rf_opt = best_rf_model.predict(X_test)
mae_rf = mean_absolute_error(y_test, y_pred_rf_opt)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf_opt))
r2_rf = r2_score(y_test, y_pred_rf_opt)

df_results.loc['Random Forest Otimizado'] = [mae_rf, rmse_rf, r2_rf]
print('Melhores parâmetros encontrados para o Random Forest:', rf_grid_search.best_params_)
display(df_results)

## 5. Visualização das Previsões e Importância das Variáveis

In [ ]:
# Gráfico de Valores Reais vs. Valores Previstos (Random Forest)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred_rf_opt, color='darkorange', alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title(f'Random Forest Otimizado (R² = {r2_rf:.3f})')
plt.xlabel('Valores Reais (MPa)')
plt.ylabel('Valores Previstos (MPa)')
plt.tight_layout()
plt.show()

# Importância das features
df_importance = pd.DataFrame({
    'Variável': X.columns,
    'Importância': best_rf_model.feature_importances_
}).sort_values(by='Importância', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='Importância', y='Variável', data=df_importance, palette='autumn', hue='Variável', legend=False)
plt.title('Importância das Variáveis - Random Forest Otimizado')
plt.xlabel('Importância Relativa')
plt.tight_layout()
plt.show()

## 6. Previsão com Novos Dados

In [ ]:
# Exemplo estruturado de nova mistura fictícia
novo_concreto = pd.DataFrame({
    'Cement (component 1)(kg in a m^3 mixture)': [320.0],
    'Blast Furnace Slag (component 2)(kg in a m^3 mixture)': [120.0],
    'Fly Ash (component 3)(kg in a m^3 mixture)': [0.0],
    'Water  (component 4)(kg in a m^3 mixture)': [180.0],
    'Superplasticizer (component 5)(kg in a m^3 mixture)': [8.0],
    'Coarse Aggregate  (component 6)(kg in a m^3 mixture)': [950.0],
    'Fine Aggregate (component 7)(kg in a m^3 mixture)': [750.0],
    'Age (day)': [28]
})

predicao = best_rf_model.predict(novo_concreto)[0]
print(f'Resistência estimada para a mistura aos 28 dias: {predicao:.2f} MPa')